In [1]:
import pandas as pd
import duckdb
from matplotlib import pyplot as plt
import numpy as np
from IPython.display import display
import seaborn as sns
import sklearn as slk
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import FunctionTransformer,StandardScaler, OneHotEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import joblib

In [2]:
customer_train=pd.read_csv(r'C:\Users\DELL\OneDrive\Desktop\MrEvansDataAnalysisClass\Classification\Customer_churn\train.csv')
customer_train

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,...,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,...,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,...,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,...,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,...,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0
4,57,16.953078,966.325422,Premium,Electronic check,Yes,TV Shows,No,TV,20.083397,...,20,Comedy,3.616170,4,Female,0,No,No,4LGYPK7VOL,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243782,77,9.639902,742.272460,Basic,Mailed check,No,Movies,No,Computer,13.502729,...,47,Sci-Fi,3.697451,1,Male,8,Yes,No,FBZ38J108Z,0
243783,117,13.049257,1526.763053,Premium,Credit card,No,TV Shows,Yes,TV,24.963291,...,35,Comedy,1.449742,4,Male,20,No,No,W4AO1Y6NAI,0
243784,113,14.514569,1640.146267,Premium,Credit card,Yes,TV Shows,No,TV,10.628728,...,44,Action,4.012217,6,Male,13,Yes,Yes,0H3SWWI7IU,0
243785,7,18.140555,126.983887,Premium,Bank transfer,Yes,TV Shows,No,TV,30.466782,...,36,Fantasy,2.135789,7,Female,5,No,Yes,63SJ44RT4A,0


In [3]:
customer_train.head()

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,...,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,...,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,...,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,...,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,...,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0
4,57,16.953078,966.325422,Premium,Electronic check,Yes,TV Shows,No,TV,20.083397,...,20,Comedy,3.616170,4,Female,0,No,No,4LGYPK7VOL,0


In [4]:
customer_test=pd.read_csv(r'C:\Users\DELL\OneDrive\Desktop\MrEvansDataAnalysisClass\Classification\Customer_churn\test.csv')
customer_test

#display(customer_train.head())
#display(customer_test.head())

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID
0,38,17.869374,679.036195,Premium,Mailed check,No,TV Shows,No,TV,29.126308,122.274031,42,Comedy,3.522724,2,Male,23,No,No,O1W6BHP6RM
1,77,9.912854,763.289768,Basic,Electronic check,Yes,TV Shows,No,TV,36.873729,57.093319,43,Action,2.021545,2,Female,22,Yes,No,LFR4X92X8H
2,5,15.019011,75.095057,Standard,Bank transfer,No,TV Shows,Yes,Computer,7.601729,140.414001,14,Sci-Fi,4.806126,2,Female,22,No,Yes,QM5GBIYODA
3,88,15.357406,1351.451692,Standard,Electronic check,No,Both,Yes,Tablet,35.586430,177.002419,14,Comedy,4.943900,0,Female,23,Yes,Yes,D9RXTK2K9F
4,91,12.406033,1128.949004,Standard,Credit card,Yes,TV Shows,Yes,Tablet,23.503651,70.308376,6,Drama,2.846880,6,Female,0,No,No,ENTCCHR1LR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104475,80,17.348236,1387.858873,Standard,Credit card,No,TV Shows,Yes,Mobile,19.189141,135.445204,35,Comedy,1.411831,7,Female,14,No,Yes,UTKREC613O
104476,20,8.275459,165.509180,Premium,Bank transfer,Yes,Movies,Yes,Mobile,30.986604,114.868640,17,Drama,2.783849,2,Male,8,Yes,No,MDB4E477PS
104477,106,18.134343,1922.240365,Basic,Mailed check,No,Movies,Yes,Computer,7.236303,109.583153,31,Comedy,2.991527,1,Male,12,No,Yes,IPDIA02ZE1
104478,46,19.774010,909.604454,Basic,Bank transfer,No,TV Shows,Yes,TV,25.809285,115.153570,1,Drama,4.998019,0,Female,12,Yes,No,ITLFTPRJGV


In [5]:
customer_test.head()

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID
0,38,17.869374,679.036195,Premium,Mailed check,No,TV Shows,No,TV,29.126308,122.274031,42,Comedy,3.522724,2,Male,23,No,No,O1W6BHP6RM
1,77,9.912854,763.289768,Basic,Electronic check,Yes,TV Shows,No,TV,36.873729,57.093319,43,Action,2.021545,2,Female,22,Yes,No,LFR4X92X8H
2,5,15.019011,75.095057,Standard,Bank transfer,No,TV Shows,Yes,Computer,7.601729,140.414001,14,Sci-Fi,4.806126,2,Female,22,No,Yes,QM5GBIYODA
3,88,15.357406,1351.451692,Standard,Electronic check,No,Both,Yes,Tablet,35.586430,177.002419,14,Comedy,4.943900,0,Female,23,Yes,Yes,D9RXTK2K9F
4,91,12.406033,1128.949004,Standard,Credit card,Yes,TV Shows,Yes,Tablet,23.503651,70.308376,6,Drama,2.846880,6,Female,0,No,No,ENTCCHR1LR


In [6]:
customer_train.shape

(243787, 21)

In [7]:
customer_test.shape

(104480, 20)

In [8]:
customer_test.columns

Index(['AccountAge', 'MonthlyCharges', 'TotalCharges', 'SubscriptionType',
       'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess',
       'DeviceRegistered', 'ViewingHoursPerWeek', 'AverageViewingDuration',
       'ContentDownloadsPerMonth', 'GenrePreference', 'UserRating',
       'SupportTicketsPerMonth', 'Gender', 'WatchlistSize', 'ParentalControl',
       'SubtitlesEnabled', 'CustomerID'],
      dtype='str')

In [9]:
customer_train.columns

Index(['AccountAge', 'MonthlyCharges', 'TotalCharges', 'SubscriptionType',
       'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess',
       'DeviceRegistered', 'ViewingHoursPerWeek', 'AverageViewingDuration',
       'ContentDownloadsPerMonth', 'GenrePreference', 'UserRating',
       'SupportTicketsPerMonth', 'Gender', 'WatchlistSize', 'ParentalControl',
       'SubtitlesEnabled', 'CustomerID', 'Churn'],
      dtype='str')

#churn is missing in test its the y its the label 
#y variable missing
#what is churn
0-not churn
1-churn
train =(243787, 21)
(104480, 20) test
the one colum meissing 20vs 21
your y problem is numeric dont make it  regression
THIS IS CLASSIFICATION PROBLEM  not regression
#balanced data vs imbalanced data
y variable has one classs greater than the other class
# binary classification
# check value count of yvariable to check if balanced or not
# is it supervidsed. yes cos it has y label
# is it regression or classification
# is it binary


In [10]:
customer_data_descriptions=pd.read_csv(r'C:\Users\DELL\OneDrive\Desktop\MrEvansDataAnalysisClass\Classification\Customer_churn\data_descriptions.csv')
customer_data_descriptions.head()

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.


In [11]:
customer_train['Churn'].value_counts()

Churn
0    199605
1     44182
Name: count, dtype: int64

run commmand to find percnetage

In [12]:
customer_train['Churn'].value_counts(normalize=True) * 100

Churn
0    81.876802
1    18.123198
Name: proportion, dtype: float64

In [13]:
customer_train.isnull().sum()

AccountAge                  0
MonthlyCharges              0
TotalCharges                0
SubscriptionType            0
PaymentMethod               0
PaperlessBilling            0
ContentType                 0
MultiDeviceAccess           0
DeviceRegistered            0
ViewingHoursPerWeek         0
AverageViewingDuration      0
ContentDownloadsPerMonth    0
GenrePreference             0
UserRating                  0
SupportTicketsPerMonth      0
Gender                      0
WatchlistSize               0
ParentalControl             0
SubtitlesEnabled            0
CustomerID                  0
Churn                       0
dtype: int64

In [14]:
customer_train.duplicated().sum()

np.int64(0)

the duplicate sits rows we dropping

In [15]:
customer_train = customer_train.drop_duplicates()
customer_train

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,...,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,...,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,...,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,...,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,...,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0
4,57,16.953078,966.325422,Premium,Electronic check,Yes,TV Shows,No,TV,20.083397,...,20,Comedy,3.616170,4,Female,0,No,No,4LGYPK7VOL,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243782,77,9.639902,742.272460,Basic,Mailed check,No,Movies,No,Computer,13.502729,...,47,Sci-Fi,3.697451,1,Male,8,Yes,No,FBZ38J108Z,0
243783,117,13.049257,1526.763053,Premium,Credit card,No,TV Shows,Yes,TV,24.963291,...,35,Comedy,1.449742,4,Male,20,No,No,W4AO1Y6NAI,0
243784,113,14.514569,1640.146267,Premium,Credit card,Yes,TV Shows,No,TV,10.628728,...,44,Action,4.012217,6,Male,13,Yes,Yes,0H3SWWI7IU,0
243785,7,18.140555,126.983887,Premium,Bank transfer,Yes,TV Shows,No,TV,30.466782,...,36,Fantasy,2.135789,7,Female,5,No,Yes,63SJ44RT4A,0


In [16]:
customer_test.duplicated().sum()

np.int64(0)

In [17]:
customer_test = customer_train.drop_duplicates()

In [18]:
customer_train = customer_train.drop_duplicates(subset=['CustomerID'])

indepen varianble-depend
correation=train.corr(numeric_only=True)
correlation['churn'].sort_values(ascending=True)

NOTE IF U HAVE 10 COLUMNS AND CORR SHOWS STRONG -VE OR STROBG +VE BETWEEN 7COLUMNS DROP THE REST.
+1 STRONG CORRELATION

STRING -VE IS -0.5 TO -1
WEAK -VE IS -O.1 TO -0.4
stratify=customer_train['Churn']
stratify=customer_train['yvariable here']]

random samling with equal data  no too much of this or that is what stratify ensures

In [19]:
correlation = customer_train.corr(numeric_only=True)

In [20]:
correlation['Churn'].sort_values(ascending=True)

AccountAge                 -0.197736
AverageViewingDuration     -0.146897
ContentDownloadsPerMonth   -0.129752
ViewingHoursPerWeek        -0.128645
TotalCharges               -0.120529
WatchlistSize               0.021739
UserRating                  0.022124
SupportTicketsPerMonth      0.084064
MonthlyCharges              0.100473
Churn                       1.000000
Name: Churn, dtype: float64

In [21]:
correlation['Churn'].sort_values(ascending=False)

Churn                       1.000000
MonthlyCharges              0.100473
SupportTicketsPerMonth      0.084064
UserRating                  0.022124
WatchlistSize               0.021739
TotalCharges               -0.120529
ViewingHoursPerWeek        -0.128645
ContentDownloadsPerMonth   -0.129752
AverageViewingDuration     -0.146897
AccountAge                 -0.197736
Name: Churn, dtype: float64

In [22]:
# Drop Churn (our target) and CustomerID (useless for training) to create X
X = customer_train.drop(columns=['Churn', 'CustomerID'])

# Grab just the Churn column for y
y = customer_train['Churn']

In [23]:
X_train, X_validation, y_train, y_validation = train_test_split(
    customer_train.drop(columns=['Churn', 'CustomerID']), 
    customer_train['Churn'], 
    test_size=0.2, 
    random_state=42,
    stratify=customer_train['Churn']
)
print(X_train.shape, X_validation.shape)

(195029, 19) (48758, 19)


#up next PREPROCESSING steps
using power transformer or standad scaler

In [24]:
# 1. Automatically grab numeric and categorical column names from X_train
numeric_cols = make_column_selector(dtype_include=['number'])(X_train)

categorical_cols = make_column_selector(dtype_include=['object', 'category'])(X_train)

# 2. Build the numeric pipeline (fill missing with median, then scale)
num_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# 3. Build the categorical pipeline (fill missing with 'missing', then encode text)
cat_transformer = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='missing'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

# 4. Combine them into one master preprocessor transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

# Let's inspect the columns to make sure they split correctly
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['AccountAge', 'MonthlyCharges', 'TotalCharges', 'ViewingHoursPerWeek', 'AverageViewingDuration', 'ContentDownloadsPerMonth', 'UserRating', 'SupportTicketsPerMonth', 'WatchlistSize']
Categorical columns: ['SubscriptionType', 'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess', 'DeviceRegistered', 'GenrePreference', 'Gender', 'ParentalControl', 'SubtitlesEnabled']


In [25]:
from sklearn.preprocessing import PowerTransformer

# 1. Automatically grab numeric and categorical column names from X_train
numeric_cols = make_column_selector(dtype_include=['number'])(X_train)
categorical_cols = make_column_selector(dtype_include=['object', 'category'])(X_train)

# 2. Build the numeric pipeline with Yeo-Johnson & Standardize=True
num_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    PowerTransformer(method='yeo-johnson', standardize=True)
)

# 3. Build the categorical pipeline (same as before)
cat_transformer = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='missing'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

# 4. Combine them into your master preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

print("Preprocessor built successfully using Yeo-Johnson strategy!")

Preprocessor built successfully using Yeo-Johnson strategy!


In [26]:
from sklearn.preprocessing import PowerTransformer

# 1. Automatically grab numeric and categorical column names
numeric_cols = make_column_selector(dtype_include=['number'])(X_train)
categorical_cols = make_column_selector(dtype_include=['object', 'category'])(X_train)

# 2. Numeric pipeline (No imputer needed, just Yeo-Johnson transformation)
num_transformer = make_pipeline(
    PowerTransformer(method='yeo-johnson', standardize=True)
)

# 3. Categorical pipeline (No imputer needed, just One-Hot Encoder)
cat_transformer = make_pipeline(
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

# 4. Combine them into your master preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

print("Clean preprocessor built without imputers!")

Clean preprocessor built without imputers!


In [27]:
#ignore the rest build without pipeline this version
from sklearn.preprocessing import PowerTransformer

# 1. Automatically grab numeric and categorical column names
numeric_cols = make_column_selector(dtype_include=['number'])(X_train)
categorical_cols = make_column_selector(dtype_include=['object', 'category'])(X_train)

# 2. Feed the transformers directly into the ColumnTransformer (No pipelines!)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='yeo-johnson', standardize=True), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

print("Preprocessor configured perfectly without make_pipeline!")

Preprocessor configured perfectly without make_pipeline!


In [28]:
!pip install imbalanced-learn

In [29]:
from imblearn.pipeline import make_pipeline as imbl_pipeline
from sklearn.linear_model import LogisticRegression

overmplin vs undersampling vs smooth . brings them on par. technique oversampling under brings unbalance one at bottom at par with top classifiers include logistic regressor note ITS FOR CLASSIFICATION NOT REGRESSOR SUPPORT VECTOR CLASSIFIER RANDOM FOREST CLASSIFIER XG BOOST CLASSIFIER EGS OF CLASSIFICATION ALGORITHMS

In [30]:
from imblearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Combine your preprocessor and the balanced Logistic Regression into the pipeline
model_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
)

# Train the entire pipeline on your stratified training data!
model_pipeline.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for fo

In [31]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# 1. Combine your preprocessor and the balanced Logistic Regression using sklearn's make_pipeline
model_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
)

# 2. Train the entire pipeline on your stratified training data!
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differe

In [32]:
from sklearn.ensemble import RandomForestClassifier

# Create and train your second pipeline using Random Forest
rf_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(class_weight='balanced', random_state=42)
)

# Train the Random Forest pipeline on your stratified training data!
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomforestclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the dif

In [33]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as imbl_pipeline
from sklearn.linear_model import LogisticRegression

# 1. Combine preprocessor, SMOTE, and Logistic Regression (no class weights!)
smote_lr_pipeline = imbl_pipeline(
    preprocessor,
    SMOTE(random_state=42),
    LogisticRegression(random_state=42, max_iter=1000)
)

# 2. Train the new SMOTE-balanced Logistic Regression model
smote_lr_pipeline.fit(X_train, y_train)

print("Logistic Regression pipeline successfully trained using SMOTE resampling!")

Logistic Regression pipeline successfully trained using SMOTE resampling!


In [34]:
#RANDOM FOREST CLASSIFIER REPEATED WITHOUT WEIGHT

In [35]:
from sklearn.ensemble import RandomForestClassifier

# 1. Combine preprocessor, SMOTE, and Random Forest (no class weights!)
smote_rf_pipeline = imbl_pipeline(
    preprocessor,
    SMOTE(random_state=42),
    RandomForestClassifier(random_state=42)
)

# 2. Train the new SMOTE-balanced Random Forest model
smote_rf_pipeline.fit(X_train, y_train)

print("Random Forest pipeline successfully trained using SMOTE resampling!")

Random Forest pipeline successfully trained using SMOTE resampling!


#BALANCED RANDOM FOREST CLASSIFIER

In [36]:
from imblearn.ensemble import BalancedRandomForestClassifier

# 1. Combine preprocessor and the Balanced Random Forest Classifier 
# (Note: Since it handles sampling internally, we drop SMOTE from the pipeline!)
brf_pipeline = imbl_pipeline(
    preprocessor,
    BalancedRandomForestClassifier(random_state=42)
)

# 2. Train your Balanced Random Forest model!
brf_pipeline.fit(X_train, y_train)

print("Balanced Random Forest pipeline successfully trained!")

Balanced Random Forest pipeline successfully trained!


In [37]:
from sklearn.metrics import classification_report

# 1. Balanced Random Forest Classifier
print("="*60)
print("1. BALANCED RANDOM FOREST CLASSIFIER")
print("="*60)
brf_preds = brf_pipeline.predict(X_validation)
print(classification_report(y_validation, brf_preds))

# 2. Random Forest Classifier with Weights
print("="*60)
print("2. RANDOM FOREST CLASSIFIER (WITH CLASS WEIGHTS)")
print("="*60)
rf_weight_preds = rf_pipeline.predict(X_validation)
print(classification_report(y_validation, rf_weight_preds))

# 3. Random Forest Classifier without Weights (SMOTE)
print("="*60)
print("3. RANDOM FOREST CLASSIFIER (WITHOUT WEIGHTS + SMOTE)")
print("="*60)
rf_smote_preds = smote_rf_pipeline.predict(X_validation)
print(classification_report(y_validation, rf_smote_preds))

# 4. Logistic Regression with Weights
print("="*60)
print("4. LOGISTIC REGRESSION (WITH CLASS WEIGHTS)")
print("="*60)
lr_weight_preds = model_pipeline.predict(X_validation)
print(classification_report(y_validation, lr_weight_preds))

# 5. Logistic Regression without Weights (SMOTE)
print("="*60)
print("5. LOGISTIC REGRESSION (WITHOUT WEIGHTS + SMOTE)")
print("="*60)
lr_smote_preds = smote_lr_pipeline.predict(X_validation)
print(classification_report(y_validation, lr_smote_preds))

1. BALANCED RANDOM FOREST CLASSIFIER
              precision    recall  f1-score   support

           0       0.88      0.81      0.84     39921
           1       0.37      0.51      0.43      8837

    accuracy                           0.76     48758
   macro avg       0.63      0.66      0.64     48758
weighted avg       0.79      0.76      0.77     48758

2. RANDOM FOREST CLASSIFIER (WITH CLASS WEIGHTS)
              precision    recall  f1-score   support

           0       0.82      0.99      0.90     39921
           1       0.59      0.04      0.08      8837

    accuracy                           0.82     48758
   macro avg       0.71      0.52      0.49     48758
weighted avg       0.78      0.82      0.75     48758

3. RANDOM FOREST CLASSIFIER (WITHOUT WEIGHTS + SMOTE)
              precision    recall  f1-score   support

           0       0.84      0.95      0.89     39921
           1       0.47      0.20      0.28      8837

    accuracy                           0.8

Best for Catching Churners: Logistic Regression (Model 4 & 5)
Class 1 Recall: 0.70 (70%)

The Good: It successfully flags 70% of all the people who are actually planning to leave the company.

The Catch: Its precision is 0.32. This means when it flags a group of people as "at risk of churning," only 32% of them actually leave (there are a lot of false alarms).

Best Overall Balance: Balanced Random Forest (Model 1)
Class 1 Recall: 0.51 (51%) | Class 1 Precision: 0.37 (37%)

The Verdict: It doesn't catch quite as many churners as Logistic Regression, but it makes significantly fewer false alarms, keeping your overall dataset cleaner.

class_report= classification_report(y_train,y_pred)
cross_val=cross_val_pred(model, y_train,x_train,cv=5)

In [38]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

# 1. Balanced Random Forest Classifier
print("="*60)
print("1. BALANCED RANDOM FOREST (CV RESULTS)")
print("="*60)
cv_brf_preds = cross_val_predict(brf_pipeline, X_train, y_train, cv=5, n_jobs=-1)
print(classification_report(y_train, cv_brf_preds))

# 2. Random Forest Classifier with Weights
print("="*60)
print("2. RANDOM FOREST (WITH CLASS WEIGHTS) (CV RESULTS)")
print("="*60)
cv_rf_weight_preds = cross_val_predict(rf_pipeline, X_train, y_train, cv=5, n_jobs=-1)
print(classification_report(y_train, cv_rf_weight_preds))

# 3. Random Forest Classifier without Weights (SMOTE)
print("="*60)
print("3. RANDOM FOREST (WITHOUT WEIGHTS + SMOTE) (CV RESULTS)")
print("="*60)
cv_rf_smote_preds = cross_val_predict(smote_rf_pipeline, X_train, y_train, cv=5, n_jobs=-1)
print(classification_report(y_train, cv_rf_smote_preds))

# 4. Logistic Regression with Weights
print("="*60)
print("4. LOGISTIC REGRESSION (WITH CLASS WEIGHTS) (CV RESULTS)")
print("="*60)
cv_lr_weight_preds = cross_val_predict(model_pipeline, X_train, y_train, cv=5, n_jobs=-1)
print(classification_report(y_train, cv_lr_weight_preds))

# 5. Logistic Regression without Weights (SMOTE)
print("="*60)
print("5. LOGISTIC REGRESSION (WITHOUT WEIGHTS + SMOTE) (CV RESULTS)")
print("="*60)
cv_lr_smote_preds = cross_val_predict(smote_lr_pipeline, X_train, y_train, cv=5, n_jobs=-1)
print(classification_report(y_train, cv_lr_smote_preds))

1. BALANCED RANDOM FOREST (CV RESULTS)
              precision    recall  f1-score   support

           0       0.88      0.81      0.84    159684
           1       0.37      0.50      0.42     35345

    accuracy                           0.75    195029
   macro avg       0.62      0.65      0.63    195029
weighted avg       0.79      0.75      0.77    195029

2. RANDOM FOREST (WITH CLASS WEIGHTS) (CV RESULTS)
              precision    recall  f1-score   support

           0       0.82      0.99      0.90    159684
           1       0.59      0.04      0.07     35345

    accuracy                           0.82    195029
   macro avg       0.70      0.52      0.49    195029
weighted avg       0.78      0.82      0.75    195029

3. RANDOM FOREST (WITHOUT WEIGHTS + SMOTE) (CV RESULTS)
              precision    recall  f1-score   support

           0       0.84      0.95      0.89    159684
           1       0.46      0.19      0.27     35345

    accuracy                        

In [39]:
# Create a dictionary pairing a clean name with each of your 5 pre-built pipeline objects
models_dict = {
    "1. BALANCED RANDOM FOREST": brf_pipeline,
    "2. RANDOM FOREST (WITH CLASS WEIGHTS)": rf_pipeline,
    "3. RANDOM FOREST (WITHOUT WEIGHTS + SMOTE)": smote_rf_pipeline,
    "4. LOGISTIC REGRESSION (WITH CLASS WEIGHTS)": model_pipeline,
    "5. LOGISTIC REGRESSION (WITHOUT WEIGHTS + SMOTE)": smote_lr_pipeline
}

# Loop through each model name and its pipeline object automatically
for name, pipeline in models_dict.items():
    print("="*60)
    print(f"{name} (CV RESULTS)")
    print("="*60)
    
    # Generate the cross-validated predictions using the current pipeline in the loop
    cv_preds = cross_val_predict(pipeline, X_train, y_train, cv=5, n_jobs=-1)
    
    # Print the report card
    print(classification_report(y_train, cv_preds))
    print("\n")  # Adds a clean blank line between reports

1. BALANCED RANDOM FOREST (CV RESULTS)
              precision    recall  f1-score   support

           0       0.88      0.81      0.84    159684
           1       0.37      0.50      0.42     35345

    accuracy                           0.75    195029
   macro avg       0.62      0.65      0.63    195029
weighted avg       0.79      0.75      0.77    195029



2. RANDOM FOREST (WITH CLASS WEIGHTS) (CV RESULTS)
              precision    recall  f1-score   support

           0       0.82      0.99      0.90    159684
           1       0.59      0.04      0.07     35345

    accuracy                           0.82    195029
   macro avg       0.70      0.52      0.49    195029
weighted avg       0.78      0.82      0.75    195029



3. RANDOM FOREST (WITHOUT WEIGHTS + SMOTE) (CV RESULTS)
              precision    recall  f1-score   support

           0       0.84      0.95      0.89    159684
           1       0.46      0.19      0.27     35345

    accuracy                    

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Loop through each model name and its pipeline object automatically
for name, pipeline in models_dict.items():
    print("="*60)
    print(f"{name} METRICS")
    print("="*60)
    
    # 1. Generate your ypredict (cross-validated predictions)
    cv_preds = cross_val_predict(pipeline, X_train, y_train, cv=5, n_jobs=-1)
    
    # 2. Calculate individual scores passing (y_train, ypredict)
    precision = precision_score(y_train, cv_preds)
    recall = recall_score(y_train, cv_preds)
    f1 = f1_score(y_train, cv_preds)
    matrix = confusion_matrix(y_train, cv_preds)
    
    # 3. Print them out beautifully
    print(f"Precision Score : {precision:.4f}")
    print(f"Recall Score    : {recall:.4f}")
    print(f"F1 Score        : {f1:.4f}")
    print("\nConfusion Matrix:")
    print(matrix)
    print("\n")

1. BALANCED RANDOM FOREST METRICS
Precision Score : 0.3688
Recall Score    : 0.4990
F1 Score        : 0.4241

Confusion Matrix:
[[129494  30190]
 [ 17708  17637]]


2. RANDOM FOREST (WITH CLASS WEIGHTS) METRICS
